<a href="https://colab.research.google.com/github/Aastharadha02/multi-agent-debugger/blob/main/Multi_Agent_Debugger_Workflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install groq


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 5.7 MB/s eta 0:00:00


In [2]:
import os
os.environ["GROQ_API_KEY"] = "Your_Groq_key"


In [3]:
import os
import json
import ast
import re
from typing import TypedDict, List, Dict, Any
from dataclasses import dataclass
from groq import Groq

# =====================================================
# SHARED STATE
# =====================================================
class AgentState(TypedDict):
    code: str
    instruction: str
    scanner_output: dict
    errors: List[str]
    language: str
    fixed_code: str
    validation: dict


# =====================================================
# SCANNER AGENT
# =====================================================
@dataclass
class Diagnostic:
    line: int
    type: str
    description: str
    severity: str


class RobustCodeScanner:
    def scan(self, code: str) -> Dict[str, Any]:
        diagnostics = []
        language = "Python" if re.search(r"\bdef\b|\bimport\b|print\(", code) else "Unknown"

        if language == "Python":
            try:
                ast.parse(code)
            except SyntaxError as e:
                diagnostics.append(
                    Diagnostic(e.lineno or 1, "syntax", e.msg, "critical")
                )

            if re.search(r"\beval\s*\(", code):
                diagnostics.append(
                    Diagnostic(0, "security", "Use of eval()", "critical")
                )

            if re.search(r"\bexec\s*\(", code):
                diagnostics.append(
                    Diagnostic(0, "security", "Use of exec()", "critical")
                )

        return {
            "language": language,
            "diagnostics": [d.__dict__ for d in diagnostics]
        }


def scanner_agent(state: AgentState) -> AgentState:
    scanner = RobustCodeScanner()
    output = scanner.scan(state["code"])

    state["scanner_output"] = output
    state["errors"] = [d["description"] for d in output["diagnostics"]]
    state["language"] = output["language"]

    return state


# =====================================================
# FIXER AGENT (GROQ)
# =====================================================
client = Groq()  # API key must be in environment variable
MODEL_NAME = "llama-3.1-8b-instant"


def clean_code(text: str) -> str:
    text = re.sub(r"```[a-zA-Z]*", "", text)
    return text.replace("```", "").strip()


def fixer_agent(state: AgentState) -> AgentState:
    prompt = f"""
Fix the code strictly according to the instruction.

Rules:
- Do not invent new functionality
- If code is already correct, return it unchanged
- Fix only what is requested
- Return ONLY corrected code

CODE:
{state["code"]}

INSTRUCTION:
{state["instruction"]}
"""

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1
    )

    state["fixed_code"] = clean_code(response.choices[0].message.content)
    return state


# =====================================================
# VALIDATOR AGENT
# =====================================================
def validator_agent(state: AgentState) -> AgentState:
    validation = {
        "syntax_ok": True,
        "logic_ok": True,
        "issues": []
    }

    # Syntax validation
    try:
        ast.parse(state["fixed_code"])
    except SyntaxError:
        validation["syntax_ok"] = False
        validation["issues"].append("Syntax error still present")

    # Logical validation via LLM
    prompt = f"""
Check whether the fixed code satisfies the instruction.

Reply ONLY with:
YES or NO

INSTRUCTION:
{state["instruction"]}

FIXED CODE:
{state["fixed_code"]}
"""

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    verdict = response.choices[0].message.content.strip().upper()

    if verdict != "YES":
        validation["logic_ok"] = False
        validation["issues"].append("Logic does not satisfy instruction")

    state["validation"] = validation
    return state


# =====================================================
# INTERACTIVE LOOP
# =====================================================
def run():
    print("MULTI-AGENT CODE DEBUGGER")

    print("\nEnter code (end with empty line):")
    lines = []
    while True:
        line = input()
        if line.strip() == "":
            break
        lines.append(line)

    code = "\n".join(lines)

    while True:
        instruction = input("\nWhat should be done with this code? ")

        state: AgentState = {
            "code": code,
            "instruction": instruction,
            "scanner_output": {},
            "errors": [],
            "language": "",
            "fixed_code": "",
            "validation": {}
        }

        # Scanner
        print("\nSCANNER AGENT OUTPUT")
        state = scanner_agent(state)
        print(json.dumps(state["scanner_output"], indent=2))

        if state["errors"]:
            print("\nDetected Errors:")
            for i, err in enumerate(state["errors"], 1):
                print(f"{i}. {err}")
        else:
            print("\nNo errors detected by scanner.")

        # Fixer
        print("\nFIXER AGENT RUNNING")
        state = fixer_agent(state)

        # Validator
        print("\nVALIDATOR AGENT RUNNING")
        state = validator_agent(state)

        # Final output
        print("\nORIGINAL CODE:\n")
        print(state["code"])

        print("\nFINAL CODE:\n")
        print(state["fixed_code"])

        print("\nVALIDATION RESULT:")
        print(json.dumps(state["validation"], indent=2))

        # User decision
        print("\nChoose next action:")
        print("1. Accept and exit")
        print("2. Accept but make more changes")
        print("3. Not satisfied, try again")

        choice = input("Enter choice (1/2/3): ").strip()

        if choice == "1":
            print("\nSession ended.")
            break
        elif choice in ("2", "3"):
            code = state["fixed_code"]
            print("\nContinuing with updated code.")
        else:
            print("\nInvalid choice. Exiting.")
            break


# =====================================================
# START
# =====================================================
run()


MULTI-AGENT CODE DEBUGGER

Enter code (end with empty line):
printf(Hello)


What should be done with this code? correct errror

SCANNER AGENT OUTPUT
{
  "language": "Unknown",
  "diagnostics": []
}

No errors detected by scanner.

FIXER AGENT RUNNING

VALIDATOR AGENT RUNNING

ORIGINAL CODE:

printf(Hello)

FINAL CODE:

printf("Hello")

VALIDATION RESULT:
{
  "syntax_ok": true,
  "logic_ok": false,
  "issues": [
    "Logic does not satisfy instruction"
  ]
}

Choose next action:
1. Accept and exit
2. Accept but make more changes
3. Not satisfied, try again
Enter choice (1/2/3): 1

Session ended.
